In [ ]:
import csv
from pathlib import Path

import pandas as pd

# 경로 및 원본 파일 설정

BASE = Path("/")
OUTPUT_DIR = BASE / "labeled"


SOURCE_FILES = {
    "logon": "logon.csv",
    "http": "http.csv",
    "file": "file.csv",
    "email": "email.csv",
    "device": "device.csv",
}

if not BASE.exists():
    raise FileNotFoundError(f"폴더가 없습니다: {BASE}")


# 1. r5.2- 로 시작하는 모든 answer CSV 읽기
# answer row format: data_type, event_id, date, user, ...

answer_files = sorted(BASE.glob("r5.2-*.csv"))

if not answer_files:
    raise FileNotFoundError(f"r5.2-로 시작하는 answer CSV가 없습니다: {BASE}")

malicious_pairs = {modality: set() for modality in SOURCE_FILES}
unknown_types = set()

print("===== Answer files =====")
for answer_path in answer_files:
    print("-", answer_path.name)

    with open(answer_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        for row in csv.reader(f):
            if len(row) < 4:
                continue

            modality = row[0].strip().lower()
            event_id = row[1].strip()
            user = row[3].strip()

            # header 처리
            if modality in {"type", "data_type", "datatype"}:
                continue

            if modality not in malicious_pairs:
                unknown_types.add(modality)
                continue

            if not event_id or not user:
                raise ValueError(
                    f"빈 id 또는 user 발견: file={answer_path.name}, row={row}"
                )

            malicious_pairs[modality].add((user, event_id))

if unknown_types:
    print("\nWARNING: 지원하지 않는 answer type:", sorted(unknown_types))

print("\n===== Answer malicious event counts =====")
for modality, pairs in malicious_pairs.items():
    print(f"{modality:>8}: {len(pairs):,}")


# 2. 원본 5개 CSV에 label 생성 및 저장

summary = []

for modality, filename in SOURCE_FILES.items():
    source_path = BASE / filename

    if not source_path.exists():
        raise FileNotFoundError(f"[{modality}] 원본 파일이 없습니다: {source_path}")

    df = pd.read_csv(
        source_path,
        dtype={"id": "string", "user": "string"},
        low_memory=False
    )

    if not {"id", "user"}.issubset(df.columns):
        raise ValueError(f"[{modality}] id 또는 user 컬럼이 없습니다.")

    df["id"] = df["id"].str.strip()
    df["user"] = df["user"].str.strip()

    if df["id"].isna().any() or df["user"].isna().any():
        raise ValueError(f"[{modality}] id 또는 user 결측값이 있습니다.")

    answer_pairs = malicious_pairs[modality]

    # (user, id)가 answer의 (user, event_id)와 일치하면 1, 아니면 0
    df["label"] = [
        int((user, event_id) in answer_pairs)
        for user, event_id in zip(df["user"], df["id"])
    ]

    # answer에 있는데 원본에서 찾지 못한 event 검증
    found_pairs = set(
        zip(
            df.loc[df["label"] == 1, "user"],
            df.loc[df["label"] == 1, "id"]
        )
    )
    missing_pairs = answer_pairs - found_pairs

    if missing_pairs:
        print(
            f"\nWARNING [{modality}]: answer event 중 원본에서 매칭되지 않은 값 "
            f"{len(missing_pairs):,}개"
        )
        print("예시:", list(missing_pairs)[:5])

    output_path = OUTPUT_DIR / f"{modality}_labeled.csv"
    df.to_csv(output_path, index=False)

    positive_events = int(df["label"].sum())
    positive_users = df.loc[df["label"] == 1, "user"].nunique()

    print(
        f"\n[{modality.upper()}] rows={len(df):,}, "
        f"users={df['user'].nunique():,}, "
        f"positive_events={positive_events:,}, "
        f"positive_users={positive_users:,}"
    )

    summary.append({
        "modality": modality,
        "rows": len(df),
        "answer_events": len(answer_pairs),
        "matched_positive_events": positive_events,
        "positive_users": positive_users,
        "unmatched_answer_events": len(missing_pairs),
    })

print("\n===== FINAL SUMMARY =====")
print(pd.DataFrame(summary).to_string(index=False))

print(f"\n완료: {OUTPUT_DIR}")